# Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

In [ ]:
df = pd.read_parquet("../data/processed/anime_data_1.parquet")
current_df = pd.read_csv("../data/raw/current_data.csv")

Let's remove duplicates.

In [ ]:
df2 = df.drop_duplicates(subset=['mal_id'], keep='first')
current_df2 = current_df.drop_duplicates(subset=['mal_id'], keep='first')

## Future Anime

In the data collection process, one of the criteria was that it should not be currently airing. This will of course include Fall 2026 and future anime. We will remove anime with release year 2027 and above, since we need Fall 2026 anime as our final prediction data.

In [ ]:
df3 = df2[df2['year'] < 2027]

current_df3 = current_df # just for tracking purposes

df3.info()

## Synopsis

No synopsis should be fine. My justification is that shows with no synopsis might be boring for users who look at MAL. However, categorizing by age rating is important: we need to know if restrictive shows have lower metrics.

In [ ]:
df4 = df3[~df3['rating'].isna()]
current_df4 = current_df3

df4.info()

Additionally, we can fill the null synopsis entries with an empty string for data entry purposes.

In [ ]:
df4.fillna({'synopsis': " "}, inplace=True)
current_df4.fillna({'synopsis': " "}, inplace=True)

df4.info()

## Multi-valued Features

If we check some multi-valued features such as genres, we can see that they're not actually lists, but strings.

In [ ]:
df4['genres']

Let's turn them into actual lists.

In [ ]:
def parse_list_col(x):
  if isinstance(x, (list, tuple, np.ndarray)):
    return list(x)

  if pd.isna(x) or x is None:
    return []

  if isinstance(x, str):
    x = x.strip()
    if not x:
      return []
    try:
      parsed = ast.literal_eval(x)
      return (
          parsed
          if isinstance(parsed, list)
          else [parsed]
          if parsed is not None
          else []
      )
    except (ValueError, SyntaxError):
      return []

  return [x]

cols = ["themes", "genres", "studios", "demographics", "producers"]
df4[cols] = df4[cols].apply(lambda col: col.map(parse_list_col))
df4['genres'] = df['genres'].apply(
    lambda genres: [g for g in genres if g != 'Award Winning']
)
current_df4[cols] = current_df4[cols].apply(lambda col: col.map(parse_list_col))
current_df4['genres'] = current_df['genres'].apply(
    lambda genres: [g for g in genres if g != 'Award Winning']
)

df4['genres']

For now, we can save this.

In [ ]:
target_dir = Path("../data/processed")
file_path1 = target_dir / "anime_data_1.parquet"
df4.to_parquet(file_path1, engine='pyarrow')

file_path2 = target_dir / "real_data_1.parquet"
current_df4.to_parquet(file_path2, engine='pyarrow')